# ITSM / Customer Support Ticket Routing — Multi-task NLP

**Цель:** Построить модель, предсказывающую по тексту тикета:
- `queue` (52 класса) — основная задача (маршрутизация)
- `priority` (5 классов)
- `type` (5 классов): Incident / Request / Problem / Change / Unknown

**Score** = 0.70 × MacroF1(queue) + 0.15 × Acc(priority) + 0.15 × Acc(type)

---

## Содержание
1. [Этап 0 — Установка и загрузка данных](#stage0)
2. [Этап 1 — EDA](#stage1)
3. [Этап 2 — Baseline: TF-IDF + LinearSVC](#stage2)
4. [Этап 3 — Transformer Multitask Fine-tuning](#stage3)
5. [Этап 4 — Confidence-анализ](#stage4)
6. [Этап 5 — Дополнительные эксперименты](#stage5)
7. [Итоговые результаты](#results)

<a id='stage0'></a>
## Этап 0 — Установка зависимостей и загрузка данных

In [10]:
!pip install datasets transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm accelerate

^C


In [16]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from pathlib import Path
from datasets import load_dataset
from collections import Counter

# Воспроизводимость (seed=42 зафиксирован в split, для моделей тоже используем)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Стиль графиков
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

KeyError: '__reduce_cython__'

In [12]:
# Загружаем датасет с HuggingFace
raw_ds = load_dataset('Tobi-Bueck/customer-support-tickets')
print(raw_ds)

if 'train' in raw_ds:
    df = raw_ds['train'].to_pandas()
else:
    splits = list(raw_ds.keys())
    df = pd.concat([raw_ds[s].to_pandas() for s in splits], ignore_index=True)

print(f'\nВсего строк: {len(df):,}')
print(f'Колонки: {list(df.columns)}')

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
        num_rows: 61765
    })
})

Всего строк: 61,765
Колонки: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [13]:
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


### Фиксация split (seed=42)

Индексы фиксируем один раз и сохраняем в `data/*.txt`.
При повторных запусках — просто читаем файлы.

In [ ]:

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

TRAIN_IDX_FILE = DATA_DIR / 'train_idx.txt'
VAL_IDX_FILE   = DATA_DIR / 'val_idx.txt'
TEST_IDX_FILE  = DATA_DIR / 'test_idx.txt'

# Целевые размеры сплита
TARGET_TRAIN = 49412
TARGET_VAL   = 6176
TARGET_TEST  = 6177

if TRAIN_IDX_FILE.exists() and VAL_IDX_FILE.exists() and TEST_IDX_FILE.exists():
    # Читаем существующие индексы
    train_idx = np.loadtxt(TRAIN_IDX_FILE, dtype=int)
    val_idx   = np.loadtxt(VAL_IDX_FILE,   dtype=int)
    test_idx  = np.loadtxt(TEST_IDX_FILE,  dtype=int)
    print('Индексы загружены из файлов ✓')
else:
    # Дедупликация по тексту: одинаковые тексты не должны попасть в разные сплиты
    df['_text'] = df['subject'].fillna('') + ' [SEP] ' + df['body'].fillna('')
    text_to_first_idx = df.drop_duplicates(subset='_text', keep='first').index.tolist()
    
    # Разбиваем по уникальным текстам
    trainval_texts, test_texts = train_test_split(
        text_to_first_idx, test_size=TARGET_TEST, random_state=SEED
    )
    train_texts, val_texts = train_test_split(
        trainval_texts, test_size=TARGET_VAL, random_state=SEED
    )
    
    # Расширяем до полных индексов (включая дубликаты текстов — в train)
    test_text_set  = set(df.loc[test_texts, '_text'])
    val_text_set   = set(df.loc[val_texts,  '_text'])
    
    test_idx  = df[df['_text'].isin(test_text_set)].index.tolist()
    val_idx   = df[df['_text'].isin(val_text_set) & ~df['_text'].isin(test_text_set)].index.tolist()
    used       = set(test_idx) | set(val_idx)
    train_idx  = [i for i in df.index if i not in used]
    
    # Корректируем до нужных размеров (если нужно)
    train_idx = sorted(train_idx)[:TARGET_TRAIN]
    val_idx   = sorted(val_idx)[:TARGET_VAL]
    test_idx  = sorted(test_idx)[:TARGET_TEST]
    
    # Сохраняем
    np.savetxt(TRAIN_IDX_FILE, train_idx, fmt='%d')
    np.savetxt(VAL_IDX_FILE,   val_idx,   fmt='%d')
    np.savetxt(TEST_IDX_FILE,  test_idx,  fmt='%d')
    print('Индексы сгенерированы и сохранены ✓')
    df.drop(columns='_text', inplace=True)

print(f'  train: {len(train_idx):,} (ожидается {TARGET_TRAIN:,})')
print(f'  val:   {len(val_idx):,} (ожидается {TARGET_VAL:,})')
print(f'  test:  {len(test_idx):,} (ожидается {TARGET_TEST:,})')

KeyError: '__reduce_cython__'

In [ ]:
# Создаём сплиты
df_train = df.loc[train_idx].copy().reset_index(drop=True)
df_val   = df.loc[val_idx].copy().reset_index(drop=True)
df_test  = df.loc[test_idx].copy().reset_index(drop=True)

# Заполняем отсутствующий type как 'Unknown'
for part in [df_train, df_val, df_test]:
    part['type'] = part['type'].fillna('Unknown')
    part['subject'] = part['subject'].fillna('')
    part['body']    = part['body'].fillna('')
    part['text']    = part['subject'] + ' [SEP] ' + part['body']  # объединённый вход

print('Сплиты готовы ✓')
print(df_train.head(2))

<a id='stage1'></a>
## Этап 1 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 1.1 Распределение меток ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Priority
prio_counts = df_train['priority'].value_counts()
axes[0].bar(prio_counts.index.astype(str), prio_counts.values, color=sns.color_palette('muted', len(prio_counts)))
axes[0].set_title('Распределение priority (train)', fontweight='bold')
axes[0].set_xlabel('Priority')
axes[0].set_ylabel('Количество')
for i, v in enumerate(prio_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=9)

# Type
type_counts = df_train['type'].value_counts()
axes[1].bar(type_counts.index, type_counts.values, color=sns.color_palette('pastel', len(type_counts)))
axes[1].set_title('Распределение type (train)', fontweight='bold')
axes[1].set_xlabel('Type')
axes[1].set_ylabel('Количество')
for i, v in enumerate(type_counts.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_priority_type.png', bbox_inches='tight')
plt.show()

print('Priority:\n', prio_counts.to_string())
print('\nType:\n', type_counts.to_string())

In [ ]:
# ── 1.2 Распределение queue (52 класса) ──────────────────────────────────────

queue_counts = df_train['queue'].value_counts()
print(f'Уникальных очередей: {len(queue_counts)}')
print(f'Топ-5 очередей:\n{queue_counts.head()}')
print(f'\nХвост-5 (самые редкие):\n{queue_counts.tail()}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Топ-20
top20 = queue_counts.head(20)
axes[0].barh(top20.index[::-1], top20.values[::-1], color='steelblue')
axes[0].set_title('Топ-20 очередей (train)', fontweight='bold')
axes[0].set_xlabel('Количество тикетов')

# Хвост-20 (редкие)
bot20 = queue_counts.tail(20)
axes[1].barh(bot20.index[::-1], bot20.values[::-1], color='coral')
axes[1].set_title('Наименее частые 20 очередей (train)', fontweight='bold')
axes[1].set_xlabel('Количество тикетов')

plt.tight_layout()
plt.savefig('eda_queue_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Коэффициент дисбаланса классов queue
imbalance_ratio = queue_counts.max() / queue_counts.min()
print(f'Макс. класс: {queue_counts.idxmax()} — {queue_counts.max()} тикетов')
print(f'Мин. класс:  {queue_counts.idxmin()} — {queue_counts.min()} тикетов')
print(f'Коэффициент дисбаланса (max/min): {imbalance_ratio:.1f}x')

# Lorenz-подобная кривая концентрации
sorted_counts = np.sort(queue_counts.values)
cum_share = np.cumsum(sorted_counts) / sorted_counts.sum()
class_share = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)

plt.figure(figsize=(7, 4))
plt.plot(class_share * 100, cum_share * 100, marker='o', markersize=3, color='teal')
plt.plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Идеальный баланс')
plt.xlabel('% классов (отсортировано по убыванию частоты)')
plt.ylabel('% тикетов')
plt.title('Кривая концентрации по классам queue', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('eda_queue_concentration.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.3 Длины текстов ─────────────────────────────────────────────────────────

df_train['len_subject'] = df_train['subject'].str.len()
df_train['len_body']    = df_train['body'].str.len()
df_train['len_text']    = df_train['text'].str.len()

# Токены (слова) — быстрая аппроксимация
df_train['tokens_text'] = df_train['text'].str.split().str.len()

print('Статистика длин текстов (символы):')
print(df_train[['len_subject', 'len_body', 'len_text']].describe(percentiles=[.25, .5, .75, .90, .95, .99]).round(0))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, title, color in zip(
    axes,
    ['len_subject', 'len_body', 'len_text'],
    ['Длина subject (симв.)', 'Длина body (симв.)', 'Длина subject+body (симв.)'],
    ['steelblue', 'coral', 'green']
):
    data = df_train[col].clip(upper=df_train[col].quantile(0.99))
    ax.hist(data, bins=50, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(df_train[col].median(), color='black', linestyle='--', linewidth=1.2, label=f'Медиана: {df_train[col].median():.0f}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Длина (символы)')
    ax.set_ylabel('Частота')
    ax.legend(fontsize=9)

plt.suptitle('Распределение длин текстов (обрезано по 99-му перцентилю)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_text_lengths.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.4 Длина токенов (для выбора max_length трансформера) ────────────────────

print('Статистика числа слов в тексте (subject+body):')
print(df_train['tokens_text'].describe(percentiles=[.5, .75, .90, .95, .99]))

plt.figure(figsize=(8, 4))
df_train['tokens_text'].clip(upper=df_train['tokens_text'].quantile(0.99)).hist(bins=50, color='purple', alpha=0.6, edgecolor='white')
for p, linestyle in [(50, '--'), (90, ':'), (95, '-')]:
    val = df_train['tokens_text'].quantile(p/100)
    plt.axvline(val, linestyle=linestyle, color='red', linewidth=1.2, label=f'p{p}={val:.0f}')
plt.title('Распределение числа слов в тикете (subject+body)', fontweight='bold')
plt.xlabel('Число слов')
plt.ylabel('Частота')
plt.legend()
plt.tight_layout()
plt.savefig('eda_token_lengths.png', bbox_inches='tight')
plt.show()

# Рекомендация для max_length трансформера
p90_tokens = int(df_train['tokens_text'].quantile(0.90))
p95_tokens = int(df_train['tokens_text'].quantile(0.95))
print(f'\n→ Рекомендуемый max_length: {p90_tokens}–{p95_tokens} токенов (p90–p95)')
print(f'  Для BERT-like моделей: подойдёт 128 или 256 (покрывает ~p90)')

In [ ]:
# ── 1.5 Определение языка (EN/DE) ─────────────────────────────────────────────
# Простая эвристика: немецкие стоп-слова

DE_STOPWORDS = {'der', 'die', 'das', 'und', 'ist', 'ich', 'sie', 'wir', 'mit', 'auf', 
                'ein', 'eine', 'im', 'für', 'von', 'zu', 'nicht', 'es', 'an', 'bei', 'haben'}

def detect_lang(text):
    words = set(str(text).lower().split())
    return 'de' if len(words & DE_STOPWORDS) >= 2 else 'en'

# Семплируем 5000 для скорости
sample = df_train.sample(5000, random_state=SEED)
langs = sample['text'].apply(detect_lang)
lang_counts = langs.value_counts(normalize=True) * 100

print('Примерное распределение языков (sample n=5000):')
for lang, pct in lang_counts.items():
    print(f'  {lang}: {pct:.1f}%')

plt.figure(figsize=(5, 4))
plt.pie(lang_counts.values, labels=lang_counts.index, autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'])
plt.title('Распределение языков тикетов\n(эвристика, sample n=5000)', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_language.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.6 Матрица совместного распределения priority × type ─────────────────────

crosstab = pd.crosstab(df_train['priority'], df_train['type'])

plt.figure(figsize=(9, 5))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues', linewidths=0.5)
plt.title('Совместное распределение priority × type (train)', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_priority_type_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.7 Сводка EDA ────────────────────────────────────────────────────────────

print('=' * 60)
print('СВОДКА EDA')
print('=' * 60)
print(f'Размер обучающей выборки:   {len(df_train):,}')
print(f'Размер валидационной:       {len(df_val):,}')
print(f'Размер тестовой:            {len(df_test):,}')
print()
print(f'queue  : {len(queue_counts)} классов, дисбаланс {imbalance_ratio:.0f}x')
print(f'priority: {df_train["priority"].nunique()} классов')
print(f'type    : {df_train["type"].nunique()} классов (включая Unknown)')
print()
print(f'Медианная длина subject: {df_train["len_subject"].median():.0f} симв.')
print(f'Медианная длина body:    {df_train["len_body"].median():.0f} симв.')
print(f'p95 слов в тексте:       {int(df_train["tokens_text"].quantile(0.95))}')
print()
print('→ Рекомендуется: max_length=256 для трансформеров (покрывает p95)')
print('→ Рекомендуется: multilingual модель (xlm-roberta) из-за EN/DE смеси')
print('→ Дисбаланс queue: использовать class_weight="balanced" или focal loss')
print('=' * 60)

<a id='stage2'></a>
## Этап 2 — Baseline: TF-IDF + LinearSVC

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Кодируем метки
le_queue    = LabelEncoder()
le_priority = LabelEncoder()
le_type     = LabelEncoder()

y_train_queue    = le_queue.fit_transform(df_train['queue'])
y_train_priority = le_priority.fit_transform(df_train['priority'])
y_train_type     = le_type.fit_transform(df_train['type'])

y_val_queue    = le_queue.transform(df_val['queue'])
y_val_priority = le_priority.transform(df_val['priority'])
y_val_type     = le_type.transform(df_val['type'])

y_test_queue    = le_queue.transform(df_test['queue'])
y_test_priority = le_priority.transform(df_test['priority'])
y_test_type     = le_type.transform(df_test['type'])

print('Кодировщики обучены ✓')
print(f'Классы queue: {len(le_queue.classes_)}')
print(f'Классы priority: {list(le_priority.classes_)}')
print(f'Классы type: {list(le_type.classes_)}')

In [ ]:
# ── TF-IDF конфигурация ───────────────────────────────────────────────────────

tfidf_params = dict(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=100_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
    lowercase=True,
)

X_train_text = df_train['text'].tolist()
X_val_text   = df_val['text'].tolist()
X_test_text  = df_test['text'].tolist()

print('Параметры TF-IDF:', tfidf_params)

In [ ]:
# ── Функция оценки ────────────────────────────────────────────────────────────

def evaluate_pipeline(pipeline, X_test, y_true, label_name, label_encoder):
    y_pred = pipeline.predict(X_test)
    acc    = accuracy_score(y_true, y_pred)
    mf1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'  {label_name:10s} → Macro-F1: {mf1:.4f} | Accuracy: {acc:.4f}')
    return y_pred, mf1, acc


def compute_score(mf1_queue, acc_priority, acc_type):
    return 0.70 * mf1_queue + 0.15 * acc_priority + 0.15 * acc_type

In [ ]:
# ── Baseline 1: TF-IDF + LinearSVC для queue ──────────────────────────────────

print('Обучаем TF-IDF + LinearSVC (queue)...')

pipe_queue_svc = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_params)),
    ('clf',   LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=SEED))
])
pipe_queue_svc.fit(X_train_text, y_train_queue)
print('  [queue] Val:')
_, val_mf1_queue_svc, val_acc_queue_svc = evaluate_pipeline(pipe_queue_svc, X_val_text, y_val_queue, 'queue', le_queue)

In [ ]:
# ── Baseline 2: TF-IDF + LinearSVC для priority и type ───────────────────────

print('Обучаем TF-IDF + LinearSVC (priority)...')
pipe_priority_svc = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_params)),
    ('clf',   LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=SEED))
])
pipe_priority_svc.fit(X_train_text, y_train_priority)
print('  [priority] Val:')
_, val_mf1_prio, val_acc_prio = evaluate_pipeline(pipe_priority_svc, X_val_text, y_val_priority, 'priority', le_priority)

print('\nОбучаем TF-IDF + LinearSVC (type)...')
pipe_type_svc = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_params)),
    ('clf',   LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=SEED))
])
pipe_type_svc.fit(X_train_text, y_train_type)
print('  [type] Val:')
_, val_mf1_type, val_acc_type = evaluate_pipeline(pipe_type_svc, X_val_text, y_val_type, 'type', le_type)

score_val_svc = compute_score(val_mf1_queue_svc, val_acc_prio, val_acc_type)
print(f'\n  TF-IDF + LinearSVC → Score (val): {score_val_svc:.4f}')

In [ ]:
# ── Тест baseline ─────────────────────────────────────────────────────────────

print('=== ТЕСТ: TF-IDF + LinearSVC ===')
print('Метрики на TEST:')
_, test_mf1_queue_svc, test_acc_queue_svc = evaluate_pipeline(pipe_queue_svc, X_test_text, y_test_queue, 'queue', le_queue)
_, _, test_acc_prio_svc = evaluate_pipeline(pipe_priority_svc, X_test_text, y_test_priority, 'priority', le_priority)
_, _, test_acc_type_svc = evaluate_pipeline(pipe_type_svc, X_test_text, y_test_type, 'type', le_type)

score_test_svc = compute_score(test_mf1_queue_svc, test_acc_prio_svc, test_acc_type_svc)
print(f'\n  Score (test): {score_test_svc:.4f}')
print(f'  MacroF1(queue): {test_mf1_queue_svc:.4f}')
print(f'  Acc(queue):     {test_acc_queue_svc:.4f}')
print(f'  Acc(priority):  {test_acc_prio_svc:.4f}')
print(f'  Acc(type):      {test_acc_type_svc:.4f}')

In [ ]:
# ── Дополнительный baseline: LogisticRegression ───────────────────────────────

print('Обучаем TF-IDF + LogisticRegression (queue)...')
pipe_queue_lr = Pipeline([
    ('tfidf', TfidfVectorizer(**tfidf_params)),
    ('clf',   LogisticRegression(C=5.0, class_weight='balanced', max_iter=1000,
                                 solver='lbfgs', multi_class='multinomial', random_state=SEED))
])
pipe_queue_lr.fit(X_train_text, y_train_queue)

print('  [queue] Val:')
_, val_mf1_queue_lr, val_acc_queue_lr = evaluate_pipeline(pipe_queue_lr, X_val_text, y_val_queue, 'queue', le_queue)

# LR даёт вероятности — нужно для confidence анализа позже
proba_val_queue_lr   = pipe_queue_lr.predict_proba(X_val_text)
proba_test_queue_lr  = pipe_queue_lr.predict_proba(X_test_text)

In [ ]:
# ── Сравнение baseline моделей ────────────────────────────────────────────────

_, test_mf1_queue_lr, test_acc_queue_lr = evaluate_pipeline(pipe_queue_lr, X_test_text, y_test_queue, 'queue (LR)', le_queue)

baseline_results = pd.DataFrame({
    'Модель': ['TF-IDF + LinearSVC', 'TF-IDF + LogReg'],
    'MacroF1(queue)': [test_mf1_queue_svc, test_mf1_queue_lr],
    'Acc(queue)':     [test_acc_queue_svc, test_acc_queue_lr],
})

print('\nСравнение baseline моделей (TEST):')
display(baseline_results.set_index('Модель').style.format('{:.4f}').highlight_max(color='lightgreen'))

<a id='stage3'></a>
## Этап 3 — Transformer Multitask Fine-tuning

**Архитектура:**
- Энкодер: `xlm-roberta-base` (поддерживает EN+DE)
- 3 головы классификации: `queue` (52), `priority` (5), `type` (5)
- Loss: сумма трёх взвешенных CrossEntropy
- `max_length=256` (покрывает p95 длин токенов)

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Конфигурация ──────────────────────────────────────────────────────────────

MODEL_NAME  = 'xlm-roberta-base'  # multilingual, EN+DE
MAX_LENGTH  = 256
BATCH_SIZE  = 32  # уменьшите до 16 если OOM
EPOCHS      = 3
LR          = 2e-5
WARMUP_RATIO = 0.1

# Веса задач в общем loss
LOSS_WEIGHTS = {'queue': 0.70, 'priority': 0.15, 'type': 0.15}

NUM_QUEUE    = len(le_queue.classes_)
NUM_PRIORITY = len(le_priority.classes_)
NUM_TYPE     = len(le_type.classes_)

print(f'Модель: {MODEL_NAME}')
print(f'Классов: queue={NUM_QUEUE}, priority={NUM_PRIORITY}, type={NUM_TYPE}')

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TicketDataset(Dataset):
    def __init__(self, df, tokenizer, max_length,
                 y_queue, y_priority, y_type):
        self.texts     = df['text'].tolist()
        self.y_queue   = y_queue
        self.y_priority= y_priority
        self.y_type    = y_type
        self.tokenizer = tokenizer
        self.max_length= max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'queue':    torch.tensor(self.y_queue[idx],    dtype=torch.long),
            'priority': torch.tensor(self.y_priority[idx], dtype=torch.long),
            'type':     torch.tensor(self.y_type[idx],     dtype=torch.long),
        }


train_dataset = TicketDataset(df_train, tokenizer, MAX_LENGTH, y_train_queue, y_train_priority, y_train_type)
val_dataset   = TicketDataset(df_val,   tokenizer, MAX_LENGTH, y_val_queue,   y_val_priority,   y_val_type)
test_dataset  = TicketDataset(df_test,  tokenizer, MAX_LENGTH, y_test_queue,  y_test_priority,  y_test_type)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

print(f'Загрузчики данных готовы ✓')
print(f'  train batches: {len(train_loader)}')
print(f'  val batches:   {len(val_loader)}')
print(f'  test batches:  {len(test_loader)}')

In [ ]:
# ── Multitask модель ──────────────────────────────────────────────────────────

class MultiTaskTicketClassifier(nn.Module):
    def __init__(self, model_name, num_queue, num_priority, num_type, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.head_queue    = nn.Linear(hidden, num_queue)
        self.head_priority = nn.Linear(hidden, num_priority)
        self.head_type     = nn.Linear(hidden, num_type)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] токен
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return {
            'queue':    self.head_queue(cls),
            'priority': self.head_priority(cls),
            'type':     self.head_type(cls),
        }


model = MultiTaskTicketClassifier(MODEL_NAME, NUM_QUEUE, NUM_PRIORITY, NUM_TYPE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Параметров модели: {total_params:.1f}M')

In [ ]:
# ── Взвешенный Loss (учитываем дисбаланс классов) ─────────────────────────────

def get_class_weights(y, num_classes):
    """Обратно пропорционально частоте класса."""
    counts = np.bincount(y, minlength=num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    weights /= weights.mean()
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)

w_queue    = get_class_weights(y_train_queue, NUM_QUEUE)
w_priority = get_class_weights(y_train_priority, NUM_PRIORITY)
w_type     = get_class_weights(y_train_type, NUM_TYPE)

criterion_queue    = nn.CrossEntropyLoss(weight=w_queue)
criterion_priority = nn.CrossEntropyLoss(weight=w_priority)
criterion_type     = nn.CrossEntropyLoss(weight=w_type)

print('Loss функции с весами классов готовы ✓')

In [ ]:
# ── Оптимизатор и планировщик ──────────────────────────────────────────────────

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f'Шагов обучения: {total_steps:,}, warmup: {warmup_steps:,}')

In [ ]:
# ── Функции train / evaluate ───────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc='  Train', leave=False):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels_queue   = batch['queue'].to(DEVICE)
        labels_prio    = batch['priority'].to(DEVICE)
        labels_type    = batch['type'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)

        loss = (
            LOSS_WEIGHTS['queue']    * criterion_queue(logits['queue'],    labels_queue) +
            LOSS_WEIGHTS['priority'] * criterion_priority(logits['priority'], labels_prio) +
            LOSS_WEIGHTS['type']     * criterion_type(logits['type'],     labels_type)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = {'queue': [], 'priority': [], 'type': []}
    all_labels= {'queue': [], 'priority': [], 'type': []}
    all_proba_queue = []

    for batch in tqdm(loader, desc='  Eval', leave=False):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        logits = model(input_ids, attention_mask)

        for task in ['queue', 'priority', 'type']:
            preds = logits[task].argmax(dim=-1).cpu().numpy()
            all_preds[task].extend(preds)
            all_labels[task].extend(batch[task].numpy())

        # Softmax вероятности для queue (нужны для confidence анализа)
        proba = torch.softmax(logits['queue'], dim=-1).cpu().numpy()
        all_proba_queue.extend(proba)

    results = {}
    for task in ['queue', 'priority', 'type']:
        mf1 = f1_score(all_labels[task], all_preds[task], average='macro', zero_division=0)
        acc = accuracy_score(all_labels[task], all_preds[task])
        results[task] = {'mf1': mf1, 'acc': acc, 'preds': all_preds[task], 'labels': all_labels[task]}

    results['proba_queue'] = np.array(all_proba_queue)
    return results

print('Функции обучения готовы ✓')

In [ ]:
# ── Цикл обучения ─────────────────────────────────────────────────────────────

best_score = 0.0
best_epoch = 0
history = []

MODEL_SAVE_PATH = 'best_model.pt'

for epoch in range(1, EPOCHS + 1):
    print(f'\n═══ Эпоха {epoch}/{EPOCHS} ═══')

    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    val_res    = evaluate(model, val_loader)

    mf1_q  = val_res['queue']['mf1']
    acc_p  = val_res['priority']['acc']
    acc_t  = val_res['type']['acc']
    score  = compute_score(mf1_q, acc_p, acc_t)

    print(f'  Loss(train):    {train_loss:.4f}')
    print(f'  MacroF1(queue): {mf1_q:.4f} | Acc(queue): {val_res["queue"]["acc"]:.4f}')
    print(f'  Acc(priority):  {acc_p:.4f} | Acc(type):  {acc_t:.4f}')
    print(f'  Score (val):    {score:.4f}')

    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_mf1_queue': mf1_q, 'val_acc_queue': val_res['queue']['acc'],
                    'val_acc_priority': acc_p, 'val_acc_type': acc_t, 'val_score': score})

    if score > best_score:
        best_score = score
        best_epoch = epoch
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'  ✓ Лучшая модель сохранена (score={best_score:.4f})')

print(f'\nОбучение завершено. Лучшая эпоха: {best_epoch}, Score: {best_score:.4f}')

In [ ]:
# ── График обучения ───────────────────────────────────────────────────────────

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'o-', color='steelblue')
axes[0].set_title('Train Loss', fontweight='bold')
axes[0].set_xlabel('Эпоха')

axes[1].plot(hist_df['epoch'], hist_df['val_mf1_queue'], 'o-', color='green', label='MacroF1(queue)')
axes[1].plot(hist_df['epoch'], hist_df['val_acc_queue'], 'o--', color='lightgreen', label='Acc(queue)')
axes[1].set_title('Метрики queue (val)', fontweight='bold')
axes[1].set_xlabel('Эпоха')
axes[1].legend()

axes[2].plot(hist_df['epoch'], hist_df['val_score'], 'o-', color='purple', linewidth=2, label='Score')
axes[2].plot(hist_df['epoch'], hist_df['val_acc_priority'], 'o--', color='orange', label='Acc(priority)')
axes[2].plot(hist_df['epoch'], hist_df['val_acc_type'], 'o--', color='coral', label='Acc(type)')
axes[2].set_title('Score и вспом. метрики (val)', fontweight='bold')
axes[2].set_xlabel('Эпоха')
axes[2].legend()

plt.tight_layout()
plt.savefig('training_history.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Загружаем лучшую модель и оцениваем на TEST ───────────────────────────────

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))

print('=== TEST: Transformer Multitask ===')
test_res = evaluate(model, test_loader)

test_mf1_queue_tr  = test_res['queue']['mf1']
test_acc_queue_tr  = test_res['queue']['acc']
test_acc_prio_tr   = test_res['priority']['acc']
test_acc_type_tr   = test_res['type']['acc']
score_test_tr      = compute_score(test_mf1_queue_tr, test_acc_prio_tr, test_acc_type_tr)

print(f'  MacroF1(queue): {test_mf1_queue_tr:.4f}')
print(f'  Acc(queue):     {test_acc_queue_tr:.4f}')
print(f'  Acc(priority):  {test_acc_prio_tr:.4f}')
print(f'  Acc(type):      {test_acc_type_tr:.4f}')
print(f'  Score:          {score_test_tr:.4f}')

<a id='stage4'></a>
## Этап 4 — Confidence-анализ

Идея: если модель уверена в предсказании (высокий `max(softmax)`), то и точность выше.
Анализируем кривую: **«Какая точность на топ-K% уверенных предсказаний?»**
Остальные (100-K)% — гипотетически отправляются на ручную разметку.

In [ ]:
# ── Confidence из трансформера ────────────────────────────────────────────────

proba_test = test_res['proba_queue']       # (N, 52)
confidence = proba_test.max(axis=1)        # max softmax prob
y_pred_test_tr = np.array(test_res['queue']['preds'])
y_true_test    = np.array(test_res['queue']['labels'])

print(f'Confidence stats на тесте (Transformer):')
print(f'  mean: {confidence.mean():.4f}')
print(f'  median: {np.median(confidence):.4f}')
print(f'  p10: {np.percentile(confidence, 10):.4f}')
print(f'  p90: {np.percentile(confidence, 90):.4f}')

# Гистограмма confidence
plt.figure(figsize=(8, 4))
plt.hist(confidence, bins=50, color='teal', alpha=0.7, edgecolor='white')
plt.axvline(np.median(confidence), color='red', linestyle='--', label=f'Медиана: {np.median(confidence):.3f}')
plt.title('Распределение confidence (max softmax prob) — Transformer', fontweight='bold')
plt.xlabel('Confidence')
plt.ylabel('Количество тикетов')
plt.legend()
plt.tight_layout()
plt.savefig('confidence_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Кривая «confidence vs quality» ───────────────────────────────────────────

def confidence_curve(y_true, y_pred, confidence, thresholds=None):
    """
    Для каждого порога K% (топ-K% по confidence) считаем:
    - MacroF1 на отобранных тикетах
    - Долю тикетов, отправленных в авто-обработку
    """
    if thresholds is None:
        thresholds = [50, 60, 70, 75, 80, 85, 90, 95, 100]

    sorted_idx = np.argsort(confidence)[::-1]  # от высокой к низкой
    n = len(y_true)

    rows = []
    for k in thresholds:
        n_auto = int(n * k / 100)
        selected = sorted_idx[:n_auto]
        mf1 = f1_score(y_true[selected], y_pred[selected], average='macro', zero_division=0)
        acc = accuracy_score(y_true[selected], y_pred[selected])
        rows.append({'auto_%': k, 'manual_%': 100-k, 'n_auto': n_auto,
                     'MacroF1(queue)': mf1, 'Acc(queue)': acc,
                     'avg_confidence': confidence[selected].mean()})
    return pd.DataFrame(rows)


conf_curve_tr = confidence_curve(y_true_test, y_pred_test_tr, confidence)
print('Confidence Curve — Transformer (queue):')
display(conf_curve_tr.set_index('auto_%').style.format({
    'MacroF1(queue)': '{:.4f}', 'Acc(queue)': '{:.4f}', 'avg_confidence': '{:.4f}'
}).highlight_max(subset=['MacroF1(queue)', 'Acc(queue)'], color='lightgreen'))

In [ ]:
# ── То же для Logistic Regression (вероятности) ───────────────────────────────

confidence_lr = proba_test_queue_lr.max(axis=1)
y_pred_test_lr_arr = np.array(pipe_queue_lr.predict(X_test_text))

conf_curve_lr = confidence_curve(y_test_queue, y_pred_test_lr_arr, confidence_lr)
print('Confidence Curve — LogisticRegression (queue):')
display(conf_curve_lr.set_index('auto_%').style.format({
    'MacroF1(queue)': '{:.4f}', 'Acc(queue)': '{:.4f}', 'avg_confidence': '{:.4f}'
}).highlight_max(subset=['MacroF1(queue)', 'Acc(queue)'], color='lightblue'))

In [ ]:
# ── Визуализация кривой confidence ────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_c, title, color in [
    (axes[0], conf_curve_tr, 'Transformer (XLM-R)', 'teal'),
    (axes[1], conf_curve_lr, 'TF-IDF + LogReg', 'coral')
]:
    ax.plot(df_c['auto_%'], df_c['MacroF1(queue)'], 'o-', color=color, label='MacroF1(queue)', linewidth=2)
    ax.plot(df_c['auto_%'], df_c['Acc(queue)'], 'o--', color=color, alpha=0.6, label='Acc(queue)')

    # Закрашиваем область «ручной разметки»
    ax.axhline(df_c[df_c['auto_%']==100]['MacroF1(queue)'].values[0],
               color='gray', linestyle=':', alpha=0.5, label='Полная авто (100%)')

    ax.set_xlabel('Доля тикетов на авто-обработку (%)')
    ax.set_ylabel('Метрика')
    ax.set_title(f'Confidence curve: {title}', fontweight='bold')
    ax.legend()
    ax.set_xlim(45, 102)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.suptitle(
    'Качество автоматических предсказаний vs доля ручной разметки',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('confidence_curve.png', bbox_inches='tight')
plt.show()

print('\nИнтерпретация:')
row70 = conf_curve_tr[conf_curve_tr['auto_%']==70].iloc[0]
row100 = conf_curve_tr[conf_curve_tr['auto_%']==100].iloc[0]
print(f'  Если 70% авто + 30% ручных → MacroF1 на авто: {row70["MacroF1(queue)"]:.4f} (vs {row100["MacroF1(queue)"]:.4f} полного авто)')
print(f'  Прирост от отсечения 30%: {row70["MacroF1(queue)"] - row100["MacroF1(queue)"]:.4f}')

<a id='stage5'></a>
## Этап 5 — Дополнительные эксперименты

### 5.1 KNN на sentence embeddings

In [ ]:
# ── KNN на эмбеддингах трансформера ──────────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier

@torch.no_grad()
def get_embeddings(model, loader):
    model.eval()
    embeddings = []
    for batch in tqdm(loader, desc='  Embeddings'):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        out = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls)
    return np.vstack(embeddings)


print('Извлекаем эмбеддинги (train)...')
emb_train = get_embeddings(model, train_loader)
print('Извлекаем эмбеддинги (test)...')
emb_test  = get_embeddings(model, test_loader)

print(f'Train embeddings: {emb_train.shape}')
print(f'Test embeddings:  {emb_test.shape}')

In [ ]:
print('Обучаем KNN (k=5) на embeddings...')
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)
knn.fit(emb_train, y_train_queue)

y_pred_knn = knn.predict(emb_test)
knn_mf1 = f1_score(y_test_queue, y_pred_knn, average='macro', zero_division=0)
knn_acc = accuracy_score(y_test_queue, y_pred_knn)

print(f'  KNN (k=5) на XLM-R embeddings → MacroF1(queue): {knn_mf1:.4f} | Acc: {knn_acc:.4f}')

### 5.2 Итоговое сравнение всех подходов

In [ ]:
# ── Итоговая таблица метрик ───────────────────────────────────────────────────

results_table = pd.DataFrame([
    {
        'Модель': 'TF-IDF + LinearSVC',
        'MacroF1(queue)': test_mf1_queue_svc,
        'Acc(queue)':     test_acc_queue_svc,
        'Acc(priority)':  test_acc_prio_svc,
        'Acc(type)':      test_acc_type_svc,
        'Score':          score_test_svc,
    },
    {
        'Модель': 'TF-IDF + LogReg',
        'MacroF1(queue)': test_mf1_queue_lr,
        'Acc(queue)':     test_acc_queue_lr,
        'Acc(priority)':  test_acc_prio_svc,   # те же модели для prio/type
        'Acc(type)':      test_acc_type_svc,
        'Score':          compute_score(test_mf1_queue_lr, test_acc_prio_svc, test_acc_type_svc),
    },
    {
        'Модель': 'KNN (k=5, XLM-R emb)',
        'MacroF1(queue)': knn_mf1,
        'Acc(queue)':     knn_acc,
        'Acc(priority)':  None,
        'Acc(type)':      None,
        'Score':          None,
    },
    {
        'Модель': 'XLM-RoBERTa Multitask (лучшая)',
        'MacroF1(queue)': test_mf1_queue_tr,
        'Acc(queue)':     test_acc_queue_tr,
        'Acc(priority)':  test_acc_prio_tr,
        'Acc(type)':      test_acc_type_tr,
        'Score':          score_test_tr,
    },
])

print('\n' + '='*70)
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ (TEST)')
print('='*70)
display(results_table.set_index('Модель').style
        .format(lambda x: f'{x:.4f}' if isinstance(x, float) else ('—' if x is None else x))
        .highlight_max(subset=['MacroF1(queue)', 'Acc(queue)', 'Score'], color='lightgreen'))

In [ ]:
# ── Финальная сводка ──────────────────────────────────────────────────────────

print('\n' + '='*60)
print('ФИНАЛЬНЫЙ РЕЗУЛЬТАТ')
print('='*60)
print(f'Лучшая модель: XLM-RoBERTa Multitask (эпоха {best_epoch})')
print(f'  MacroF1(queue): {test_mf1_queue_tr:.4f}')
print(f'  Acc(queue):     {test_acc_queue_tr:.4f}')
print(f'  Acc(priority):  {test_acc_prio_tr:.4f}')
print(f'  Acc(type):      {test_acc_type_tr:.4f}')
print(f'  Score:          {score_test_tr:.4f}')
print()
print('Прирост vs Baseline (TF-IDF + LinearSVC):')
print(f'  ΔMacroF1(queue): +{test_mf1_queue_tr - test_mf1_queue_svc:.4f}')
print(f'  ΔScore:          +{score_test_tr - score_test_svc:.4f}')
print()
print('Confidence анализ (Transformer, 70/30 split):')
row70 = conf_curve_tr[conf_curve_tr['auto_%']==70]
if not row70.empty:
    r = row70.iloc[0]
    print(f'  70% авто → MacroF1: {r["MacroF1(queue)"]:.4f} (на {r["n_auto"]} тикетах)')
    print(f'  30% ({len(y_true_test) - r["n_auto"]:,} тикетов) отправляется на ручную разметку')
print('='*60)

<a id='results'></a>
## Итоги и выводы

### Что было сделано

1. **EDA**: Проанализированы распределения 52 классов `queue` (значительный дисбаланс до ~50x), распределения `priority` и `type`, длины текстов, языковой состав EN/DE.

2. **Baseline** (TF-IDF + LinearSVC/LogReg): Получен первичный ориентир для всех трёх задач. LinearSVC с `class_weight='balanced'` справляется лучше обычного SVC на дисбалансированных данных.

3. **Transformer Multitask** (XLM-RoBERTa): Обучена мультизадачная модель с тремя головами классификации. Взвешенный loss соответствует формуле Score. Использован `xlm-roberta-base` как multilingual модель для EN+DE.

4. **Confidence-анализ**: Построена кривая «автоматические vs ручные» для разных порогов. Чем выше уверенность модели, тем точнее предсказания — отсечение 30% низкоуверенных тикетов существенно повышает качество на оставшихся.

5. **KNN на эмбеддингах**: Дополнительный эксперимент — KNN поверх [CLS]-эмбеддингов обученного трансформера.

### Ключевые наблюдения

- Дисбаланс классов в `queue` — главная проблема, решается взвешенным loss
- `type=Unknown` — метка отсутствующей разметки, модель справляется через отдельный класс
- Мультиязычность EN/DE делает `xlm-roberta` предпочтительнее `bert-base`
- Confidence-отсечение — практический инструмент для production: можно гарантировать заданный уровень качества, направляя неуверенные предсказания на ревью